# SMPL Reconstruction Lab · Phase 2

Chạy HMR2.0b pretrained với một ảnh PandaSet. Cấu hình ban đầu chọn cảnh `006`,
khung `03`, camera `front_camera`.

Trong **Thời gian chạy → Thay đổi loại thời gian chạy**, chọn **GPU** và
phiên bản Colab **2026.07 (Python 3.12)**. Sau đó chạy các ô theo thứ tự và
cấp quyền gắn Drive. Notebook cần ảnh RGB đã chọn và bản SMPL neutral
được cấp phép. Lần đầu chạy sẽ tải khoảng 2,71 GB checkpoint và dữ liệu
từ 4D-Humans vào bộ nhớ của runtime. Kết quả được lưu trong thư mục riêng
của bạn trên Drive.

[4D-Humans](https://github.com/shubham-goel/4D-Humans) ·
[Phiên bản Colab](https://research.google.com/colaboratory/runtime-version-faq.html)

## 1. Kiểm tra GPU và gắn Drive

In [ ]:
import sys
from hashlib import file_digest
import json
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Chọn phiên bản runtime Colab 2026.07 (Python 3.12), rồi chạy lại.")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chọn GPU trong mục Thời gian chạy → Thay đổi loại thời gian chạy.")
print(f"GPU: {torch.cuda.get_device_name(0)} | PyTorch {torch.__version__}")

from google.colab import drive
drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/SMPL Reconstruction Lab - PandaSet Phase 2")
FRAMES = PROJECT / "front_camera - frames da chon"
SCENE_ID, FRAME_ID, CAMERA_ID = "006", "03", "front_camera"
PHOTO = FRAMES / f"{SCENE_ID}_{CAMERA_ID}_{FRAME_ID}.jpg"
SMPL = PROJECT / "basicModel_neutral_lbs_10_207_0_v1.0.0.pkl"
for path in (PHOTO, SMPL):
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu tệp đầu vào: {path}")
catalog_path = FRAMES / "frames.json"
if catalog_path.is_file():
    for item in json.loads(catalog_path.read_text(encoding="utf-8"))["frames"]:
        if item["file"] == PHOTO.name:
            with PHOTO.open("rb") as stream:
                if file_digest(stream, "sha256").hexdigest() != item["sha256"]:
                    raise ValueError(f"Ảnh trên Drive không khớp với ảnh PandaSet đã chọn: {PHOTO.name}")
            break
print(f"Đã tìm thấy {PHOTO.name} và mô hình SMPL neutral.")

## 2. Cài 4D-Humans và kiểm tra khả năng render

In [ ]:
import os
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/buide03/SMPL.git"
REPO = Path("/content/smpl-reconstruction-lab")
VENV = Path("/content/smpl-lab-phase2-venv")
PYTHON = VENV / "bin" / "python"

def command(args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)

if not REPO.exists():
    command(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, REPO])
else:
    origin = command(["git", "remote", "get-url", "origin"], cwd=REPO,
                     capture_output=True, text=True).stdout.strip()
    if origin != REPO_URL:
        raise RuntimeError(f"Thư mục {REPO} thuộc repo khác.")
    command(["git", "pull", "--ff-only"], cwd=REPO)

if not PYTHON.exists():
    command([sys.executable, "-m", "venv", "--system-site-packages", VENV])
package_pins = command(
    [sys.executable, "-c", "from colab.hmr2_assets import UPSTREAM, REVISION, CHUMPY_REVISION;"
     "print(UPSTREAM, REVISION, CHUMPY_REVISION)"],
    cwd=REPO, text=True, capture_output=True,
).stdout.split()
upstream, hmr_revision, chumpy_revision = package_pins
constraints = VENV / "colab-torch-constraints.txt"
from importlib.metadata import version
constraints.write_text(f"torch=={version('torch')}\ntorchvision=={version('torchvision')}\n",
                       encoding="utf-8")
command([PYTHON, "-m", "pip", "install", "setuptools>=68", "wheel"])
command([PYTHON, "-m", "pip", "install", "-c", constraints,
         "-r", REPO / "colab/requirements-colab.txt"])
command([PYTHON, "-m", "pip", "install", "--no-build-isolation", "--no-deps",
         f"chumpy @ git+https://github.com/mattloper/chumpy.git@{chumpy_revision}",
         f"hmr2 @ git+{upstream}@{hmr_revision}"])

RENDER_ENV = {**os.environ, "PYOPENGL_PLATFORM": "egl"}
test = [PYTHON, "-c", "import pyrender; r = pyrender.OffscreenRenderer(32, 32); r.delete()"]
probe = subprocess.run(test, env=RENDER_ENV, capture_output=True, text=True)
if probe.returncode != 0:
    mesa = Path("/usr/share/glvnd/egl_vendor.d/50_mesa.json")
    if not mesa.is_file():
        command(["apt-get", "update", "-qq"])
        command(["apt-get", "install", "-y", "-qq", "libegl1", "libegl-mesa0", "libgl1-mesa-dri"])
    RENDER_ENV.update({"__EGL_VENDOR_LIBRARY_FILENAMES": str(mesa),
                       "EGL_PLATFORM": "surfaceless", "LIBGL_ALWAYS_SOFTWARE": "1"})
    command(test, env=RENDER_ENV)
command([PYTHON, "-c",
         "import torch, torchvision, chumpy; from hmr2.models import load_hmr2;"
         "assert torch.cuda.is_available(), 'CUDA was lost during setup';"
         "print('CUDA và 4D-Humans đã sẵn sàng')"], env=RENDER_ENV)
command(["git", "rev-parse", "HEAD"], cwd=REPO)

## 3. Phát hiện người, chạy HMR2 và xuất ZIP

In [ ]:
from uuid import uuid4

RUN = PROJECT / "reconstruction-results" / (
    f"{SCENE_ID}_{CAMERA_ID}_{FRAME_ID}_{uuid4().hex[:8]}"
)
command([
    PYTHON, "-m", "colab.run_hmr2", "--source", PHOTO, "--smpl-model", SMPL,
    "--scene-id", SCENE_ID, "--frame-id", FRAME_ID, "--camera-id", CAMERA_ID,
    "--threshold", "0.50", "--min-people", "2", "--batch-size", "1",
    "--output-dir", RUN,
], cwd=REPO, env=RENDER_ENV)
ARCHIVE = RUN / "reconstruction.zip"
if not ARCHIVE.is_file():
    raise FileNotFoundError(f"Không tìm thấy kết quả: {ARCHIVE}")

## 4. Xem kết quả và đồng bộ Drive

In [ ]:
from IPython.display import Image, display
from zipfile import ZipFile

with ZipFile(ARCHIVE) as archive:
    manifest = json.loads(archive.read("manifest.json"))
    print(f"Kết quả: {len(manifest['people'])} người | {ARCHIVE.relative_to(PROJECT)}")
    display(Image(data=archive.read("overlay.png"), width=960))
    display(Image(data=archive.read("side_view.png")))
drive.flush_and_unmount()
print("Đã đồng bộ Drive. Tải reconstruction.zip và nhập tệp trong Streamlit.")